In [52]:

import pandas as pd
import numpy as np

In [ ]:

# ── Configuration ─────────────────────────────────────────────────────────────
TIN_FILE     = "../data/morat_baseline/Tin.dat"
TOUT_FILE    = "../data/schiffenen/T_out.dat"
OUTPUT_FILE  = "../data/morat_deep_connection/Tin_deep_noheader.dat"
DEPTH       = '-30.000'
NEW_COL_NAME = "Temp" # name for the new column in output
# ─────────────────────────────────────────────────────────────────────────────




# ── Parameters ────────────────────────────────────────────────────────────────
YEAR_LEN    = 365.25
DAY_END     = 17000
RESAMPLE_DT = 6 / 24          # 6h in days = 0.25
# ──────────────────────────────────────────────────────────────────────────────



# ── 1. Read Tin.dat ──────────────────────────────────────────────────────────
tin = pd.read_csv(
    TIN_FILE,
    sep=r"\s+",
    skiprows=3,
    header=None,
    names=["Time_d", "Depth_m", "Deep_Inflows"],  # Tin.dat has 3 data columns
)

# ── 2. Read T_out.dat ────────────────────────────────────────────────────────
tout_raw = pd.read_csv(TOUT_FILE, sep=",", header=0)



# 1. Pull Datetime + target depth, drop NaNs
df = tout_raw[['Datetime', DEPTH]].dropna().copy()

# 2. Resample to 6h grid: snap each Datetime to nearest 6h bin, then mean
df['Datetime_6h'] = (df['Datetime'] / RESAMPLE_DT).round() * RESAMPLE_DT
df = df.groupby('Datetime_6h')[DEPTH].mean().reset_index()
df.columns = ['Datetime', DEPTH]

print(f"Rows after 6h resampling: {len(df)}  (was hourly → now every {RESAMPLE_DT*24:.0f}h)")

# 3. Fractional day-of-year, snapped to 6h grid
df['doy'] = df['Datetime'] % YEAR_LEN
df['doy'] = (df['doy'] / RESAMPLE_DT).round() * RESAMPLE_DT
df['doy'] = df['doy'] % YEAR_LEN

# 4. Climatology: mean over all years for each doy bin
clim = (df.groupby('doy')[DEPTH]
          .mean()
          .rename('T_clim')
          .sort_index())

print(f"Climatology bins: {len(clim)}  (expect ~{int(YEAR_LEN/RESAMPLE_DT)} for full year)")

# 5. Build extended time axis 0 → DAY_END at 6h resolution
time_ext = np.arange(0, DAY_END + RESAMPLE_DT, RESAMPLE_DT)

# 6. Wrap-safe interpolation
doy_ext         = time_ext % YEAR_LEN
doy_ext_snapped = (doy_ext / RESAMPLE_DT).round() * RESAMPLE_DT % YEAR_LEN

x = clim.index.values
y = clim.values
x_wrap = np.concatenate([x - YEAR_LEN, x, x + YEAR_LEN])
y_wrap  = np.concatenate([y, y, y])

T_ext = np.interp(doy_ext_snapped, x_wrap, y_wrap)

# 7. Result
result = pd.DataFrame({
    'Datetime' : time_ext,
    'doy'      : doy_ext,
    'T_clim'   : T_ext
})

print(result)


In [60]:


# 1. Pull Datetime + target depth, drop NaNs
df = tout_raw[['Datetime', DEPTH]].dropna().copy()

# 2. Resample to 6h grid: snap each Datetime to nearest 6h bin, then mean
df['Datetime_6h'] = (df['Datetime'] / RESAMPLE_DT).round() * RESAMPLE_DT
df = df.groupby('Datetime_6h')[DEPTH].mean().reset_index()
df.columns = ['Datetime', DEPTH]

print(f"Rows after 6h resampling: {len(df)}  (was hourly → now every {RESAMPLE_DT*24:.0f}h)")

# 3. Fractional day-of-year, snapped to 6h grid
df['doy'] = df['Datetime'] % YEAR_LEN
df['doy'] = (df['doy'] / RESAMPLE_DT).round() * RESAMPLE_DT
df['doy'] = df['doy'] % YEAR_LEN

# 4. Climatology: mean over all years for each doy bin
clim = (df.groupby('doy')[DEPTH]
          .mean()
          .rename('T_clim')
          .sort_index())

print(f"Climatology bins: {len(clim)}  (expect ~{int(YEAR_LEN/RESAMPLE_DT)} for full year)")

# 5. Build extended time axis 0 → DAY_END at 6h resolution
time_ext = np.arange(0, DAY_END + RESAMPLE_DT, RESAMPLE_DT)

# 6. Wrap-safe interpolation
doy_ext         = time_ext % YEAR_LEN
doy_ext_snapped = (doy_ext / RESAMPLE_DT).round() * RESAMPLE_DT % YEAR_LEN

x = clim.index.values
y = clim.values
x_wrap = np.concatenate([x - YEAR_LEN, x, x + YEAR_LEN])
y_wrap  = np.concatenate([y, y, y])

T_ext = np.interp(doy_ext_snapped, x_wrap, y_wrap)

# 7. Result
result = pd.DataFrame({
    'Datetime' : time_ext,
    'doy'      : doy_ext,
    'T_clim'   : T_ext
})

print(result)

tin['new'] = np.interp(
    tin["Time_d"].values,
    result['Datetime'].values,
    result['T_clim'].values,
)


# ── 4. Write output ───────────────────────────────────────────────────────────
print(f"\nWriting '{OUTPUT_FILE}' ...")

with open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for _, row in tin.iterrows():
        fout.write(
            f"{row['Time_d']:>10.4f}"
            f"{row['Depth_m']:>11.2f}"
            f"{row['Deep_Inflows']:>18.2f}"
            f"{row['new']:>10.2f}"
            "\n"
        )

print(f"Done! Saved to '{OUTPUT_FILE}'.")




Rows after 6h resampling: 6149  (was hourly → now every 6h)
Climatology bins: 1461  (expect ~1461 for full year)
       Datetime     doy     T_clim
0          0.00    0.00   5.128610
1          0.25    0.25   5.117360
2          0.50    0.50   5.118209
3          0.75    0.75   5.118962
4          1.00    1.00   5.112553
...         ...     ...        ...
67996  16999.00  197.50  13.559443
67997  16999.25  197.75  12.253354
67998  16999.50  198.00  12.255947
67999  16999.75  198.25  12.269100
68000  17000.00  198.50  12.271934

[68001 rows x 3 columns]

Writing '../data/morat_deep_connection/Tin_deep_noheader.dat' ...
Done! Saved to '../data/morat_deep_connection/Tin_deep_noheader.dat'.


In [14]:
"""
merge_column_Qin.py
-------------------
Reads Qin.dat, removes the 3 header lines, and adds a constant column of 30
as the 4th column.

Qin.dat format:
  - Row 1: column headers  (Time [d], Depth [m], ...)
  - Row 2: metadata line
  - Row 3: metadata line
  - Row 4+: whitespace-delimited data
"""


# ── Configuration ─────────────────────────────────────────────────────────────
QIN_FILE    = "../data/morat_baseline/Qin.dat"
OUTPUT_FILE = "../data/morat_deep_connection/Qin_deep_noheader.dat"
CONSTANT    = 30
# ─────────────────────────────────────────────────────────────────────────────

qin = pd.read_csv(
    QIN_FILE,
    sep=r"\s+",
    skiprows=3,
    header=None,
    names=["Time_d", "Depth_m", "Deep_Inflows"],
)
print(f"Qin.dat: {len(qin):,} rows | Time [d] {qin['Time_d'].min():.4f} → {qin['Time_d'].max():.4f}")

qin["Constant"] = CONSTANT

with open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for _, row in qin.iterrows():
        fout.write(
            f"{row['Time_d']:>10.4f}"
            f"{row['Depth_m']:>11.2f}"
            f"{row['Deep_Inflows']:>18.2f}"
            f"{row['Constant']:>10}\n"
        )

print(f"Done! Saved to '{OUTPUT_FILE}'.")

Qin.dat: 395,833 rows | Time [d] 1.0000 → 16494.0000
Done! Saved to '../data/morat_deep_connection/Qin_deep_noheader.dat'.
